<a href="https://colab.research.google.com/github/entin2346/Tugas-4-Sistem-temu-kembali/blob/main/240210500016_Fitri_Afrianti_TF_IDF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Tugas 4

No. 2: Pembobotan (Term Weighting) TF-IDF

Nama : Fitri Afrianti

NIM  : 240210500016

In [14]:
# ---------- SEL 1: Import Library ----------
!pip install PySastrawi

import re                                   # regular expression untuk cleaning
import math                                 # fungsi log untuk perhitungan manual
import pandas as pd                         # tabel
from sklearn.feature_extraction.text import TfidfVectorizer
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# Pengaturan tampilan tabel pandas
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

In [15]:
# ---------- SEL 2: Dataset (4 dokumen pendek, masing-masing 2 kalimat) ----------
dokumen = [
    "Sistem komputer terdiri dari perangkat keras dan perangkat lunak yang bekerja sama. "
    "Prosesor adalah komponen utama perangkat keras.",

    "Jaringan komputer menghubungkan beberapa komputer agar dapat saling berbagi data. "
    "Router dan switch adalah perangkat jaringan yang penting.",

    "Kecerdasan buatan memungkinkan komputer belajar dari data. "
    "Machine learning adalah cabang kecerdasan buatan yang banyak digunakan.",

    "Sistem temu kembali informasi membantu pengguna mencari dokumen yang relevan. "
    "Pembobotan TF-IDF digunakan untuk menentukan kata penting dalam dokumen.",
]
nama_dok = [f"Dok{i}" for i in range(1, len(dokumen) + 1)]

for n, d in zip(nama_dok, dokumen):
    print(f"{n}: {d}\n")

Dok1: Sistem komputer terdiri dari perangkat keras dan perangkat lunak yang bekerja sama. Prosesor adalah komponen utama perangkat keras.

Dok2: Jaringan komputer menghubungkan beberapa komputer agar dapat saling berbagi data. Router dan switch adalah perangkat jaringan yang penting.

Dok3: Kecerdasan buatan memungkinkan komputer belajar dari data. Machine learning adalah cabang kecerdasan buatan yang banyak digunakan.

Dok4: Sistem temu kembali informasi membantu pengguna mencari dokumen yang relevan. Pembobotan TF-IDF digunakan untuk menentukan kata penting dalam dokumen.



In [16]:
# ---------- SEL 3: Preprocessing sederhana ----------
# Case folding + cleaning + tokenisasi + stopwords removal (Sastrawi)
STOPWORDS = set(StopWordRemoverFactory().get_stop_words())

def preprocess(text):
    text = text.lower()                        # 1. case folding
    text = re.sub(r"[^a-z\s]", " ", text)      # 2. cleaning (hapus angka, tanda baca, simbol)
    tokens = text.split()                      # 3. tokenisasi
    tokens = [t for t in tokens if t not in STOPWORDS]   # 4. stopwords removal
    return tokens

tokens_docs = [preprocess(d) for d in dokumen]

for n, t in zip(nama_dok, tokens_docs):
    print(f"{n}: {t}")

Dok1: ['sistem', 'komputer', 'perangkat', 'keras', 'perangkat', 'lunak', 'prosesor', 'komponen', 'utama', 'perangkat', 'keras']
Dok2: ['jaringan', 'komputer', 'menghubungkan', 'komputer', 'berbagi', 'data', 'router', 'switch', 'perangkat', 'jaringan']
Dok3: ['kecerdasan', 'buatan', 'komputer', 'belajar', 'data', 'machine', 'learning', 'cabang', 'kecerdasan', 'buatan']
Dok4: ['sistem', 'temu', 'informasi', 'membantu', 'pengguna', 'mencari', 'dokumen', 'relevan', 'pembobotan', 'tf', 'idf', 'menentukan', 'dokumen']


In [17]:
# ---------- SEL 4: Bag-of-Words (raw count) ----------
vocab = sorted(set(t for doc in tokens_docs for t in doc))   # vocabulary (term unik)
N = len(tokens_docs)                                          # jumlah dokumen

print(f"Ukuran vocabulary : {len(vocab)} term")
print(f"Jumlah dokumen (N): {N}\n")

bow = pd.DataFrame(
    [[doc.count(term) for term in vocab] for doc in tokens_docs],
    index=nama_dok, columns=vocab
)

print("Matriks Bag-of-Words (baris = term, kolom = dokumen):")
print(bow.T.astype(int))

Ukuran vocabulary : 31 term
Jumlah dokumen (N): 4

Matriks Bag-of-Words (baris = term, kolom = dokumen):
               Dok1  Dok2  Dok3  Dok4
belajar           0     0     1     0
berbagi           0     1     0     0
buatan            0     0     2     0
cabang            0     0     1     0
data              0     1     1     0
dokumen           0     0     0     2
idf               0     0     0     1
informasi         0     0     0     1
jaringan          0     2     0     0
kecerdasan        0     0     2     0
keras             2     0     0     0
komponen          1     0     0     0
komputer          1     2     1     0
learning          0     0     1     0
lunak             1     0     0     0
machine           0     0     1     0
membantu          0     0     0     1
mencari           0     0     0     1
menentukan        0     0     0     1
menghubungkan     0     1     0     0
pembobotan        0     0     0     1
pengguna          0     0     0     1
perangkat         3  

In [18]:
# ---------- SEL 5: Perhitungan manual - Term Frequency (TF) ----------
# TF(t,d) = jumlah kemunculan term t di dokumen d (raw count)
tf = bow.copy()

print("Term Frequency (TF) - raw count:")
print(tf.T.astype(int))

Term Frequency (TF) - raw count:
               Dok1  Dok2  Dok3  Dok4
belajar           0     0     1     0
berbagi           0     1     0     0
buatan            0     0     2     0
cabang            0     0     1     0
data              0     1     1     0
dokumen           0     0     0     2
idf               0     0     0     1
informasi         0     0     0     1
jaringan          0     2     0     0
kecerdasan        0     0     2     0
keras             2     0     0     0
komponen          1     0     0     0
komputer          1     2     1     0
learning          0     0     1     0
lunak             1     0     0     0
machine           0     0     1     0
membantu          0     0     0     1
mencari           0     0     0     1
menentukan        0     0     0     1
menghubungkan     0     1     0     0
pembobotan        0     0     0     1
pengguna          0     0     0     1
perangkat         3     1     0     0
prosesor          1     0     0     0
relevan          

In [19]:
# ---------- SEL 6: Perhitungan manual - Document Frequency (df) & IDF ----------
# df(t)  = jumlah dokumen yang mengandung term t
# IDF klasik     : log10(N / df)
# IDF smooth     : ln((1 + N) / (1 + df)) + 1   -> rumus yang dipakai scikit-learn
df_ = (bow > 0).sum(axis=0)

idf_klasik = df_.apply(lambda d: math.log10(N / d))
idf_smooth = df_.apply(lambda d: math.log((1 + N) / (1 + d)) + 1)

tabel_idf = pd.DataFrame({
    "df": df_,
    "N/df": N / df_,
    "IDF klasik = log10(N/df)": idf_klasik,
    "IDF smooth (sklearn)": idf_smooth,
})
print("Tabel df dan IDF:")
print(tabel_idf)

Tabel df dan IDF:
               df   N/df  IDF klasik = log10(N/df)  IDF smooth (sklearn)
belajar         1 4.0000                    0.6021                1.9163
berbagi         1 4.0000                    0.6021                1.9163
buatan          1 4.0000                    0.6021                1.9163
cabang          1 4.0000                    0.6021                1.9163
data            2 2.0000                    0.3010                1.5108
dokumen         1 4.0000                    0.6021                1.9163
idf             1 4.0000                    0.6021                1.9163
informasi       1 4.0000                    0.6021                1.9163
jaringan        1 4.0000                    0.6021                1.9163
kecerdasan      1 4.0000                    0.6021                1.9163
keras           1 4.0000                    0.6021                1.9163
komponen        1 4.0000                    0.6021                1.9163
komputer        3 1.3333         

In [20]:
# ---------- SEL 7: Perhitungan manual - TF-IDF ----------
# (a) TF-IDF klasik: TF x log10(N/df)   (tanpa normalisasi)
tfidf_klasik = tf.mul(idf_klasik, axis=1)
print("TF-IDF manual (rumus klasik):")
print(tfidf_klasik.T)

# (b) TF-IDF gaya scikit-learn: TF x IDF smooth, lalu normalisasi L2 per dokumen
tfidf_mentah = tf.mul(idf_smooth, axis=1)
norma = tfidf_mentah.pow(2).sum(axis=1).pow(0.5)          # norma Euclidean tiap dokumen
tfidf_manual = tfidf_mentah.div(norma, axis=0)            # bagi dengan norma (L2)

print("\nTF-IDF manual (IDF smooth + normalisasi L2) - dipakai untuk dibandingkan dengan sklearn:")
print(tfidf_manual.T)

TF-IDF manual (rumus klasik):
                Dok1   Dok2   Dok3   Dok4
belajar       0.0000 0.0000 0.6021 0.0000
berbagi       0.0000 0.6021 0.0000 0.0000
buatan        0.0000 0.0000 1.2041 0.0000
cabang        0.0000 0.0000 0.6021 0.0000
data          0.0000 0.3010 0.3010 0.0000
dokumen       0.0000 0.0000 0.0000 1.2041
idf           0.0000 0.0000 0.0000 0.6021
informasi     0.0000 0.0000 0.0000 0.6021
jaringan      0.0000 1.2041 0.0000 0.0000
kecerdasan    0.0000 0.0000 1.2041 0.0000
keras         1.2041 0.0000 0.0000 0.0000
komponen      0.6021 0.0000 0.0000 0.0000
komputer      0.1249 0.2499 0.1249 0.0000
learning      0.0000 0.0000 0.6021 0.0000
lunak         0.6021 0.0000 0.0000 0.0000
machine       0.0000 0.0000 0.6021 0.0000
membantu      0.0000 0.0000 0.0000 0.6021
mencari       0.0000 0.0000 0.0000 0.6021
menentukan    0.0000 0.0000 0.0000 0.6021
menghubungkan 0.0000 0.6021 0.0000 0.0000
pembobotan    0.0000 0.0000 0.0000 0.6021
pengguna      0.0000 0.0000 0.0000 0.6021
pera

In [21]:
# ---------- SEL 8: Contoh langkah demi langkah (1 term) ----------
term, dok = "perangkat", "Dok1"
tf_val   = tf.loc[dok, term]
df_val   = df_[term]
idf_val  = idf_smooth[term]

print(f"Term '{term}' pada {dok}")
print(f"  TF                = {tf_val}")
print(f"  df                = {df_val}  (dari N = {N})")
print(f"  IDF smooth        = ln((1+{N})/(1+{df_val})) + 1 = {idf_val:.4f}")
print(f"  TF x IDF          = {tf_val} x {idf_val:.4f} = {tf_val*idf_val:.4f}")
print(f"  Norma L2 {dok}    = {norma[dok]:.4f}")
print(f"  TF-IDF ternormalisasi = {tf_val*idf_val:.4f} / {norma[dok]:.4f} = {tf_val*idf_val/norma[dok]:.4f}")

Term 'perangkat' pada Dok1
  TF                = 3
  df                = 2  (dari N = 4)
  IDF smooth        = ln((1+4)/(1+2)) + 1 = 1.5108
  TF x IDF          = 3 x 1.5108 = 4.5325
  Norma L2 Dok1    = 7.3280
  TF-IDF ternormalisasi = 4.5325 / 7.3280 = 0.6185


In [22]:
# ---------- SEL 9: TF-IDF dengan scikit-learn (TfidfVectorizer) ----------
# Input: dokumen yang sudah dipreprocessing (token digabung kembali menjadi string)
corpus_bersih = [" ".join(t) for t in tokens_docs]

vectorizer = TfidfVectorizer()                     # parameter default
X = vectorizer.fit_transform(corpus_bersih)

tfidf_sklearn = pd.DataFrame(X.toarray(), index=nama_dok,
                             columns=vectorizer.get_feature_names_out())

print("Matriks TF-IDF hasil scikit-learn (baris = term, kolom = dokumen):")
print(tfidf_sklearn.T)

Matriks TF-IDF hasil scikit-learn (baris = term, kolom = dokumen):
                Dok1   Dok2   Dok3   Dok4
belajar       0.0000 0.0000 0.2770 0.0000
berbagi       0.0000 0.3033 0.0000 0.0000
buatan        0.0000 0.0000 0.5541 0.0000
cabang        0.0000 0.0000 0.2770 0.0000
data          0.0000 0.2391 0.2184 0.0000
dokumen       0.0000 0.0000 0.0000 0.5230
idf           0.0000 0.0000 0.0000 0.2615
informasi     0.0000 0.0000 0.0000 0.2615
jaringan      0.0000 0.6065 0.0000 0.0000
kecerdasan    0.0000 0.0000 0.5541 0.0000
keras         0.5230 0.0000 0.0000 0.0000
komponen      0.2615 0.0000 0.0000 0.0000
komputer      0.1669 0.3871 0.1768 0.0000
learning      0.0000 0.0000 0.2770 0.0000
lunak         0.2615 0.0000 0.0000 0.0000
machine       0.0000 0.0000 0.2770 0.0000
membantu      0.0000 0.0000 0.0000 0.2615
mencari       0.0000 0.0000 0.0000 0.2615
menentukan    0.0000 0.0000 0.0000 0.2615
menghubungkan 0.0000 0.3033 0.0000 0.0000
pembobotan    0.0000 0.0000 0.0000 0.2615
pengguna 

In [23]:
# ---------- SEL 10: Perbandingan manual vs scikit-learn ----------
print("Vocabulary sama persis  :", list(tfidf_sklearn.columns) == vocab)

selisih = (tfidf_manual - tfidf_sklearn).abs()
print(f"Selisih absolut maksimum: {selisih.max().max():.2e}")
print("Hasil manual = sklearn ? :", selisih.max().max() < 1e-9)

# Contoh perbandingan beberapa term
pembanding = pd.DataFrame({
    "Manual (Dok1)":  tfidf_manual.loc["Dok1", ["perangkat", "keras", "komputer", "sistem"]],
    "sklearn (Dok1)": tfidf_sklearn.loc["Dok1", ["perangkat", "keras", "komputer", "sistem"]],
})
print("\nContoh perbandingan pada Dok1:")
print(pembanding)

Vocabulary sama persis  : True
Selisih absolut maksimum: 1.11e-16
Hasil manual = sklearn ? : True

Contoh perbandingan pada Dok1:
           Manual (Dok1)  sklearn (Dok1)
perangkat         0.6185          0.6185
keras             0.5230          0.5230
komputer          0.1669          0.1669
sistem            0.2062          0.2062


In [24]:
# ---------- SEL 11: Term dengan bobot TF-IDF tertinggi per dokumen ----------
print("3 term teratas per dokumen (TF-IDF scikit-learn / manual L2):")
for n in nama_dok:
    top = tfidf_sklearn.loc[n].sort_values(ascending=False).head(3)
    print(f"  {n}: " + ", ".join(f"{k} ({v:.4f})" for k, v in top.items()))

print("\n3 term teratas per dokumen (TF-IDF manual klasik):")
for n in nama_dok:
    top = tfidf_klasik.loc[n].sort_values(ascending=False).head(3)
    print(f"  {n}: " + ", ".join(f"{k} ({v:.4f})" for k, v in top.items()))

3 term teratas per dokumen (TF-IDF scikit-learn / manual L2):
  Dok1: perangkat (0.6185), keras (0.5230), komponen (0.2615)
  Dok2: jaringan (0.6065), komputer (0.3871), berbagi (0.3033)
  Dok3: buatan (0.5541), kecerdasan (0.5541), belajar (0.2770)
  Dok4: dokumen (0.5230), idf (0.2615), informasi (0.2615)

3 term teratas per dokumen (TF-IDF manual klasik):
  Dok1: keras (1.2041), perangkat (0.9031), komponen (0.6021)
  Dok2: jaringan (1.2041), berbagi (0.6021), menghubungkan (0.6021)
  Dok3: buatan (1.2041), kecerdasan (1.2041), belajar (0.6021)
  Dok4: dokumen (1.2041), idf (0.6021), informasi (0.6021)


Term dengan bobot TF-IDF tertinggi pada Dok1 adalah perangkat (0,6185), Dok2 jaringan (0,6065), Dok3 buatan dan kecerdasan (sama-sama 0,5541), dan Dok4 dokumen (0,5230). Term-term ini penting karena muncul berulang di dokumennya sendiri (TF tinggi: perangkat 3 kali di Dok1, jaringan 2 kali di Dok2, buatan dan kecerdasan masing-masing 2 kali di Dok3, dokumen 2 kali di Dok4) dan sekaligus jarang muncul di dokumen lain (IDF tinggi), sehingga paling mewakili topik masing-masing dokumen: perangkat keras, jaringan, kecerdasan buatan, dan pencarian dokumen. Sebaliknya, komputer muncul di 3 dari 4 dokumen sehingga IDF-nya paling rendah (1,2231) dan bobotnya ditekan, padahal kata ini frekuensinya cukup tinggi di Dok2. Perlu dicatat bahwa pada rumus klasik log10(N/df) tanpa normalisasi, term teratas Dok1 adalah keras (1,2041) dan bukan perangkat (0,9031); perbedaan ini terjadi karena IDF smooth scikit-learn memberi selisih IDF yang lebih kecil antara term df=1 dan df=2 sehingga TF yang besar (3) lebih berpengaruh. Hasil perhitungan manual dengan rumus yang sama dengan scikit-learn (IDF smooth + normalisasi L2) identik dengan keluaran TfidfVectorizer, sehingga implementasi manual terbukti benar.